In [1]:
!wget -nc https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/edgar_allan_poe.txt
!wget -nc https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/robert_frost.txt

--2025-09-29 13:55:02--  https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/edgar_allan_poe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26622 (26K) [text/plain]
Saving to: ‘edgar_allan_poe.txt’

edgar_allan_poe.txt 100%[===================>]  26.00K  --.-KB/s    in 0.001s  

2025-09-29 13:55:02 (44.3 MB/s) - ‘edgar_allan_poe.txt’ saved [26622/26622]

--2025-09-29 13:55:02--  https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/robert_frost.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP re

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.model_selection import train_test_split

In [3]:
input_files= ['edgar_allan_poe.txt','robert_frost.txt']

In [4]:
# to see what is inside of the text files
!head robert_frost.txt

Two roads diverged in a yellow wood,
And sorry I could not travel both
And be one traveler, long I stood
And looked down one as far as I could
To where it bent in the undergrowth; 

Then took the other, as just as fair,
And having perhaps the better claim
Because it was grassy and wanted wear,
Though as for that the passing there


In [5]:
!head edgar_allan_poe.txt

LO! Death hath rear'd himself a throne
In a strange city, all alone,
Far down within the dim west
Where the good, and the bad, and the worst, and the best,
Have gone to their eternal rest.
 
There shrines, and palaces, and towers
Are not like any thing of ours
Oh no! O no! ours never loom
To heaven with that ungodly gloom!


In [6]:
# collect data into lists
# https://chatgpt.com/c/68da9503-81ac-832a-a3da-56f0161e4f55
input_texts = []
# will hold a numeric label for each line of text, based on which file it came from
labels = []


for label, f in enumerate(input_files):
  print(f"{f} corresponds to label {label}")

  for line in open(f):
    line = line.rstrip().lower()
    if line:
      # remove punctuation
      line = line.translate(str.maketrans('', '', string.punctuation))

      input_texts.append(line)
      labels.append(label)

edgar_allan_poe.txt corresponds to label 0
robert_frost.txt corresponds to label 1


In [8]:
train_text, test_text, y_train, y_test = train_test_split(input_texts, labels)

In [9]:
len(y_train), len(y_test)

(1615, 539)

In [10]:
train_text[:5]

['and such few people',
 'this he sat listening to till she gave judgment',
 'that hangs like chains of pearl on hermon hill',
 'what it is to him',
 'almost the moment he was given an opening']

In [11]:
y_train[0:5]

[1, 1, 0, 1, 1]

In [12]:
# convert text inot integers
idx = 1
word2idx = {'<unk>': 0}

word2idx is a dictionary (a mapping from keys to values).

Here it maps words → numbers (indices).

Specifically:

The special token '<unk>' (short for unknown word) is mapped to index 0.

This means: if a word doesn’t exist in your vocabulary, it will be replaced by index 0.

In [13]:
# populate word2idx
for text in train_text:
  tokens= text.split()
  for token in tokens:
    if token not in word2idx:
      word2idx[token] = idx
      idx += 1

In [16]:
#word2idx
# remove # if you want to see word2idx

In [15]:
len(word2idx)

2525

In [18]:
# convert data into integer format
train_text_int = []
test_text_int = []

for text in train_text:
  tokens = text.split()
  line_as_int =[word2idx[token] for token in tokens]
  train_text_int.append(line_as_int)

for text in test_text:
  tokens = text.split()
  line_as_int =[word2idx.get(token, 0) for token in tokens]
  test_text_int.append(line_as_int)

In [21]:
'''
another way of writing above code
line_as_int = []
for token in tokens:
    idx = word2idx.get(token, 0)   # look up token, use 0 if not found
    line_as_int.append(idx)
'''

'\nanother way of writing above code\nline_as_int = []\nfor token in tokens:\n    idx = word2idx.get(token, 0)   # look up token, use 0 if not found\n    line_as_int.append(idx)\n'

In [22]:
train_text_int[100:105]

[[9, 381, 28, 382, 383, 219, 18, 24],
 [1, 384, 385, 386, 28, 56, 18, 28, 387],
 [1, 388, 209, 28, 356],
 [389, 179, 222, 390],
 [1, 57, 391, 24, 30, 392, 305]]

https://chatgpt.com/c/68da9c4a-931c-832a-8a5b-3f40af5adbce

Markov models are applied when a system evolves over time, and the next state depends only on the current state (memoryless property).

In [24]:
# to represent the Markov model we build the A and pi matrices
# initialize A nd pi matrices - for both classes (we have A and pi for each class)
V = len(word2idx)

A0 = np.ones((V, V))
pi0 = np.ones(V)

A1 = np.ones((V, V))
pi1 = np.ones(V)


In [26]:
# compute counts for A and pi
def compute_counts(text_as_int, A, pi):
  for tokens in text_as_int:
    # last_idx tracks the previous word in the sequence
    last_idx = None
    for idx in tokens:
      if last_idx is None:
        # it's the first word in a sentence
        pi[idx] += 1
      else:
        # the last word exists, so count a transition
        A[last_idx, idx] += 1

      # update last idx
      last_idx = idx


compute_counts([t for t, y in zip(train_text_int, y_train) if y == 0], A0, pi0)
compute_counts([t for t, y in zip(train_text_int, y_train) if y == 1], A1, pi1)

zip(train_text_int, Ytrain) pairs each training sentence (train_text_int) with its label (Ytrain).

if y == 0 → select sentences labeled 0.

if y == 1 → select sentences labeled 1.

So:

A0, pi0 → transition & initial counts for class 0.

A1, pi1 → transition & initial counts for class 1.

In [27]:
# normalize A and pi so they are valid probability matrices
# convince yourself that this is equivalent to the formulas shown before
A0 /= A0.sum(axis=1, keepdims=True)
pi0 /= pi0.sum()

A1 /= A1.sum(axis=1, keepdims=True)
pi1 /= pi1.sum()

In [28]:
# log A and pi since we don't need the actual probs
logA0 = np.log(A0)
logpi0 = np.log(pi0)

logA1 = np.log(A1)
logpi1 = np.log(pi1)

In [30]:
# compute priors

# number of samples labeled class 0
count0 = sum(y == 0 for y in y_train)
# number of samples labeled class 1
count1 = sum(y == 1 for y in y_train)
total = len(y_train)
p0 = count0 / total
p1 = count1 / total
logp0 = np.log(p0)
logp1 = np.log(p1)
p0, p1

(0.34179566563467495, 0.6582043343653251)

𝑝0 =0.331 → About 33% of your training samples belong to class 0.

𝑝1 =0.669 → About 67% of your training samples belong to class 1.

So, class 1 is about twice as frequent as class 0 in your training data.

Class imbalance exists

Your dataset is not evenly distributed between classes.

Class 1 is roughly 2x more common than class 0.

Impact on classification

In a Bayesian classifier, these priors affect the posterior probabilities:

P(y∣x)∝P(y)⋅P(x∣y)

Since P(y=1) is larger, the model will lean toward class 1 unless the evidence strongly supports class 0.

Practical implications

If this imbalance reflects the real-world distribution, that’s fine (the priors should match reality).

If the dataset is artificially imbalanced, you might consider balancing techniques (resampling, class weights).

In [29]:
'''
count0 = 0
for y in y_train:
    if y == 0:
        count0 += 1

'''

'\ncount0 = 0\nfor y in y_train:\n    if y == 0:\n        count0 += 1\n\n'

In [31]:
# build a classifier
class Classifier:
  def __init__(self, logAs, logpis, logpriors):
    self.logAs = logAs
    self.logpis = logpis
    self.logpriors = logpriors
    self.K = len(logpriors) # number of classes

  def _compute_log_likelihood(self, input_, class_):
    logA = self.logAs[class_]
    logpi = self.logpis[class_]

    last_idx = None
    logprob = 0
    for idx in input_:
      if last_idx is None:
        # it's the first token
        logprob += logpi[idx]
      else:
        logprob += logA[last_idx, idx]

      # update last_idx
      last_idx = idx

    return logprob

  def predict(self, inputs):
    predictions = np.zeros(len(inputs))
    for i, input_ in enumerate(inputs):
      posteriors = [self._compute_log_likelihood(input_, c) + self.logpriors[c] \
             for c in range(self.K)]
      pred = np.argmax(posteriors)
      predictions[i] = pred
    return predictions

In [32]:
# each array must be in order since classes are assumed to index these lists
clf = Classifier([logA0, logA1], [logpi0, logpi1], [logp0, logp1])

In [34]:
Ptrain = clf.predict(train_text_int)
print(f"Train acc: {np.mean(Ptrain == y_train)}")

Train acc: 0.9969040247678018


In [36]:
Ptest = clf.predict(test_text_int)
print(f"Test acc: {np.mean(Ptest == y_test)}")

Test acc: 0.8404452690166976


In [37]:
from sklearn.metrics import confusion_matrix, f1_score

In [39]:
cm = confusion_matrix(y_train, Ptrain)
cm

array([[ 547,    5],
       [   0, 1063]])

In [40]:
cm_test = confusion_matrix(y_test, Ptest)
cm_test

array([[ 97,  69],
       [ 17, 356]])

In [41]:
f1_score(y_train, Ptrain)

0.997653683716565

In [42]:
f1_score(y_test, Ptest)

0.8922305764411027

The F1 score is about balance. It answers: “How well does my model catch positives without raising too many false alarms?”